In [ ]:
import pandas as pd
import os

# === CONFIG ===
INPUT_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output"
OUTPUT_PATH = os.path.join(INPUT_DIR, "7.3-Project_List_3rdAttempt_ShallowC.csv")

# === LOAD FILES ===
yml_df = pd.read_csv(os.path.join(INPUT_DIR, "7.1-YML_List_3nd_attempt_ShallowC.csv"))
api_summary_df = pd.read_csv(os.path.join(INPUT_DIR, "7.2-Project_List_API_3rdAttempt_ShallowC.csv"))
api_details_df = pd.read_csv(os.path.join(INPUT_DIR, "7.2-Project_List_API_Details_3rdAttempt_ShallowC.csv"))

def classify_test_type(test_type):
    test_type = str(test_type).strip().lower()
    
    if test_type in {"github_emulator_full", "github_emulator_compact", "github_gmd"}:
        return "GitHub Hosted"
    elif test_type in {"gitlab ci_emulator_manual", "circlci_emulator_manual", "travis ci_emulator_manual"}:
        return "Other CI Platform"
    elif test_type in {"firebase_full", "firebase_compact", "browerstack", "appcenter"}:
        return "Third Party Service"
    else:
        return "Other"



# === PREPARE GROUPED TEST TYPE COLUMN ===

yml_df["Test_Type_Group"] = yml_df["test_type"].apply(classify_test_type)


# === Define GitHub Action based on test_type ===
def detect_github_action_from_test_type(test_type_series):
    def has_github_type(test_type_str):
        if pd.isna(test_type_str):
            return False
        test_types = [t.strip().lower() for t in str(test_type_str).split(',')]
        return any(t.startswith("github") for t in test_types)
    return test_type_series.apply(has_github_type)

yml_df['GitHub Action'] = detect_github_action_from_test_type(yml_df['test_type'])

# === Define Third_Party and Third_Party Test based on test_type only ===
def detect_third_party(test_type_str):
    if pd.isna(test_type_str):
        return False
    test_types = [t.strip().lower() for t in str(test_type_str).split(',')]
    return any(t not in {'none', 'other'} and not t.startswith('github') for t in test_types)

def extract_third_party_tests(test_type_str):
    if pd.isna(test_type_str):
        return ''
    test_types = [t.strip() for t in str(test_type_str).split(',')]
    return ', '.join(sorted(set(
        t for t in test_types if t and not t.lower().startswith('github') and t.lower() not in {'none', 'other'}
    )))

yml_df['Third_Party'] = yml_df['test_type'].apply(detect_third_party)
yml_df['Third_Party Test'] = yml_df['test_type'].apply(extract_third_party_tests)

# === GitHub Action Test Aggregation ===
github_tests = yml_df[yml_df['GitHub Action']]
github_test_types = github_tests.groupby('full_name')['test_type'].apply(
    lambda x: ', '.join(sorted({
        str(t).strip() for t in ', '.join(x.dropna().astype(str)).split(',')
        if t and str(t).strip().lower() not in {'none', 'nan', 'other'}
    }))
).reset_index(name='GitHub Action Test')

# === Create API presence matrix ===
api_details_df['API_COL'] = 'API_' + api_details_df['api_level'].astype(str)
api_presence = pd.crosstab(api_details_df['full_name'], api_details_df['API_COL']).astype(bool)
api_presence['Total API'] = api_presence.sum(axis=1)

# === CI Platform Aggregation ===
ci_platforms_all = yml_df.groupby('full_name')['ci_platform']\
    .apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='CI_Platform')

ci_platforms_instr_only = yml_df[yml_df['instrumentation_test']].groupby('full_name')['ci_platform']\
    .apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='CI_Platform_Modified')

# === Project-level summary ===
yml_summary = yml_df.groupby('full_name').agg({
    'unit_test': 'any',
    'instrumentation_test': 'any',
    'GitHub Action': 'any',
    'Third_Party': 'any',
    'Third_Party Test': lambda x: ', '.join(sorted(set(x) - {''})) if any(x) else ''
}).reset_index().rename(columns={
    'unit_test': 'Unit Test',
    'instrumentation_test': 'Instrumentation Testing'
})

# === Test Type Grouping ===
grouped_test_types = yml_df.groupby('full_name')['Test_Type_Group'].apply(
    lambda x: ', '.join(sorted(set(x)))
).reset_index(name='Test_Type_Group')

# === Merge all together ===
merged = yml_summary.merge(api_summary_df[['full_name', 'username', 'project_name']], on='full_name', how='left')
merged = merged.merge(api_presence, on='full_name', how='left').infer_objects(copy=False)
merged = merged.merge(api_summary_df[['full_name', 'yml_count', 'yaml_errors']], on='full_name', how='left').fillna(0)
merged = merged.merge(ci_platforms_all, on='full_name', how='left')
merged = merged.merge(ci_platforms_instr_only, on='full_name', how='left')
merged = merged.merge(github_test_types, on='full_name', how='left')
merged = merged.merge(grouped_test_types, on='full_name', how='left')

# === Final export ===
output_cols = [
    'username', 'project_name', 'full_name',
    'Unit Test', 'Instrumentation Testing', 'GitHub Action', 'GitHub Action Test',
    'Third_Party', 'Third_Party Test', 'Test_Type_Group',
    'CI_Platform', 'CI_Platform_Modified'
] + sorted([col for col in api_presence.columns if col.startswith("API_")]) + ['Total API', 'yml_count', 'yaml_errors']

final_df = merged[output_cols]
final_df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Final project list saved to: {OUTPUT_PATH}")


✅ Final project list saved to: F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output\7.3-Project_List_3rdAttempt_ShallowC.csv
